In [ ]:
# ============================================
# YOLO MODEL OPTIMIZATION
# Environment Setup
# ============================================

!nvidia-smi

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Sat Sep 26 13:59:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ============================================
# INSTALL OPTIMIZATION LIBRARIES
# ============================================

!pip install -q ultralytics==8.4.27 torch-pruning==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.2 MB/s eta 0:00:00


In [ ]:
# ============================================
# VERIFY OPTIMIZATION ENVIRONMENT
# ============================================

import torch
import ultralytics
import torch_pruning as tp
import importlib.metadata

print("========== ENVIRONMENT ==========")
print("PyTorch       :", torch.__version__)
print("Ultralytics   :", ultralytics.__version__)
print("Torch-Pruning :", importlib.metadata.version("torch-pruning"))
print("CUDA          :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("CUDA version  :", torch.version.cuda)

print("=================================")

========== ENVIRONMENT ==========
PyTorch       : 2.11.0+cu128
Ultralytics   : 8.4.27
Torch-Pruning : 1.6.0
CUDA          : True
GPU           : Tesla T4
CUDA version  : 12.8


In [ ]:
# ============================================
# UPLOAD ORIGINAL BASELINE MODEL
# ============================================

from google.colab import files
import os

print("Select your ORIGINAL best.pt file")
uploaded = files.upload()

uploaded_name = list(uploaded.keys())[0]

# Keep a clear, fixed name inside Colab
baseline_model_path = "/content/baseline_best.pt"

os.rename(f"/content/{uploaded_name}", baseline_model_path)

print("\nModel uploaded successfully!")
print("Path:", baseline_model_path)
print("Size:", os.path.getsize(baseline_model_path) / (1024 * 1024), "MB")

Select your ORIGINAL best.pt file


Saving best.pt to best.pt

Model uploaded successfully!
Path: /content/baseline_best.pt
Size: 49.606889724731445 MB


In [ ]:
# ============================================
# LOAD BASELINE MODEL
# ============================================

from ultralytics import YOLO
import torch

MODEL_PATH = "/content/baseline_best.pt"

model = YOLO(MODEL_PATH)

print("========== BASELINE MODEL ==========")
print("Model loaded successfully")
print("Device:", "CUDA (T4)" if torch.cuda.is_available() else "CPU")

print("\nModel architecture:")
print(model.model)

print("\n====================================")

========== BASELINE MODEL ==========
Model loaded successfully
Device: CUDA (T4)

Model architecture:
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.00

In [ ]:
# ============================================
# BASELINE MODEL COMPLEXITY
# ============================================

import torch_pruning as tp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_torch = model.model.to(device)
model_torch.eval()

# 640x640 input — same resolution as our original baseline
example_inputs = torch.randn(1, 3, 640, 640).to(device)

# Count parameters and MACs
macs, params = tp.utils.count_ops_and_params(
    model_torch,
    example_inputs
)

print("========== T4 BASELINE COMPLEXITY ==========")
print(f"Parameters : {params:,}")
print(f"Parameters : {params / 1e6:.3f} M")
print(f"MACs       : {macs:,.0f}")
print(f"MACs       : {macs / 1e9:.3f} G")
print(f"GFLOPs*    : {(macs * 2) / 1e9:.3f} G")
print("============================================")
print("* GFLOPs shown here use 2 FLOPs per MAC.")

========== T4 BASELINE COMPLEXITY ==========
Parameters : 25,856,899
Parameters : 25.857 M
MACs       : 39,439,458,000
MACs       : 39.439 G
GFLOPs*    : 78.879 G
* GFLOPs shown here use 2 FLOPs per MAC.


In [ ]:
# ============================================
# T4 BASELINE INFERENCE SPEED
# ============================================

import time

# Warm-up
for _ in range(20):
    _ = model.predict(
        source=example_inputs,
        imgsz=640,
        device=0,
        verbose=False
    )

# Timed inference
torch.cuda.synchronize()
start = time.perf_counter()

for _ in range(100):
    _ = model.predict(
        source=example_inputs,
        imgsz=640,
        device=0,
        verbose=False
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

avg_time = elapsed / 100
fps = 1 / avg_time

print("========== T4 BASELINE SPEED ==========")
print(f"Average inference time : {avg_time * 1000:.2f} ms/image")
print(f"FPS                    : {fps:.2f}")
print("========================================")

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normali

In [ ]:
!pip install -q torch-pruning

In [ ]:
import importlib.metadata

print("Torch-Pruning version:", importlib.metadata.version("torch-pruning"))

import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch-Pruning version: 1.6.0
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from ultralytics import YOLO
import torch

MODEL_PATH = "/content/baseline_best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = YOLO(MODEL_PATH)

print("Model loaded successfully")
print("Device:", device)
print("Number of classes:", model.model.nc)
print("Number of layers:", len(model.model.model))

# Show the main architecture
print(model.model)

Model loaded successfully
Device: cuda
Number of classes: 1
Number of layers: 23
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, aff

In [ ]:
import os

print("Files in /content:")
for f in os.listdir("/content"):
    print(f)

Files in /content:
.config
baseline_best.pt
sample_data


In [ ]:
import torch
import torch_pruning as tp

# Total parameters
total_params = sum(p.numel() for p in model.model.parameters())

print("========== BASELINE MODEL CHECK ==========")
print(f"Total parameters: {total_params:,}")
print(f"Total parameters (M): {total_params / 1e6:.3f} M")

print("\n========== PRUNABLE C2f LAYERS ==========")

c2f_layers = []

for name, module in model.model.named_modules():
    if module.__class__.__name__ == "C2f":
        c2f_layers.append((name, module))
        print(
            f"{name} | "
            f"input channels: {module.cv1.conv.in_channels} | "
            f"output channels: {module.cv2.conv.out_channels}"
        )

print(f"\nTotal C2f layers found: {len(c2f_layers)}")

print("\n========== DETECTION LAYER ==========")

for name, module in model.model.named_modules():
    if module.__class__.__name__ == "Detect":
        print(f"Detect layer found: {name}")
        print(module)

========== BASELINE MODEL CHECK ==========
Total parameters: 25,856,899
Total parameters (M): 25.857 M

========== PRUNABLE C2f LAYERS ==========
model.2 | input channels: 96 | output channels: 96
model.4 | input channels: 192 | output channels: 192
model.6 | input channels: 384 | output channels: 384
model.8 | input channels: 576 | output channels: 576
model.12 | input channels: 960 | output channels: 384
model.15 | input channels: 576 | output channels: 192
model.18 | input channels: 576 | output channels: 384
model.21 | input channels: 960 | output channels: 576

Total C2f layers found: 8

========== DETECTION LAYER ==========
Detect layer found: model.22
Detect(
  (cv2): ModuleList(
    (0): Sequential(
      (0): Conv(
        (conv): Conv2d(192, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): C

In [3]:
!pip install -q ultralytics==8.4.27 torch-pruning==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.5 MB/s eta 0:00:00


In [4]:
import torch
import ultralytics
import torch_pruning
import importlib.metadata

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Ultralytics:", ultralytics.__version__)
print("Torch-Pruning:", importlib.metadata.version("torch-pruning"))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Ultralytics: 8.4.27
Torch-Pruning: 1.6.0


In [5]:
# ============================================
# LOAD BASELINE MODEL
# ============================================

from ultralytics import YOLO
import torch

MODEL_PATH = "/content/baseline_best.pt"

model = YOLO(MODEL_PATH)

print("Baseline model loaded successfully")
print("Model path:", MODEL_PATH)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Total parameters:", sum(p.numel() for p in model.model.parameters()))

FileNotFoundError: [Errno 2] No such file or directory: '/content/baseline_best.pt'

In [6]:
from google.colab import files

uploaded = files.upload()

Saving best.pt to best.pt


In [7]:
import shutil
import os

if os.path.exists("/content/best.pt"):
    shutil.copy(
        "/content/best.pt",
        "/content/baseline_best.pt"
    )

print("Baseline model prepared:")
print(os.path.exists("/content/baseline_best.pt"))

Baseline model prepared:
True


In [9]:
# ============================================
# LOAD AND VERIFY BASELINE MODEL
# ============================================

from ultralytics import YOLO
import torch

MODEL_PATH = "/content/baseline_best.pt"

model = YOLO(MODEL_PATH)

total_params = sum(
    p.numel() for p in model.model.parameters()
)

print("Baseline model loaded successfully")
print("Model path:", MODEL_PATH)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Total parameters:", total_params)
print("Total parameters (M):", total_params / 1e6)

Baseline model loaded successfully
Model path: /content/baseline_best.pt
Device: cuda
Total parameters: 25856899
Total parameters (M): 25.856899


In [10]:
# ============================================
# CREATE SAFE PRUNING COPY
# ============================================

import copy
import torch

# Deep copy = pruning experiments will NOT modify the baseline model
prune_model = copy.deepcopy(model.model)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

prune_model = prune_model.to(device)
prune_model.eval()

print("Original baseline model : SAFE")
print("Pruning copy created    : YES")
print("Device                  :", device)

print(
    "Pruning copy parameters :",
    sum(p.numel() for p in prune_model.parameters())
)

Original baseline model : SAFE
Pruning copy created    : YES
Device                  : cuda
Pruning copy parameters : 25856899


In [11]:
# ============================================
# INSPECT PRUNING COPY
# ============================================

print("C2f layers in pruning copy:")

c2f_count = 0

for name, module in prune_model.named_modules():
    if module.__class__.__name__ == "C2f":
        c2f_count += 1
        print(
            f"{name} | "
            f"in_channels={module.cv1.conv.in_channels} | "
            f"out_channels={module.cv2.conv.out_channels}"
        )

print()
print("Total C2f layers:", c2f_count)

print()
print("Last model layer:")
print(prune_model.model[-1])

C2f layers in pruning copy:
model.2 | in_channels=96 | out_channels=96
model.4 | in_channels=192 | out_channels=192
model.6 | in_channels=384 | out_channels=384
model.8 | in_channels=576 | out_channels=576
model.12 | in_channels=960 | out_channels=384
model.15 | in_channels=576 | out_channels=192
model.18 | in_channels=576 | out_channels=384
model.21 | in_channels=960 | out_channels=576

Total C2f layers: 8

Last model layer:
Detect(
  (cv2): ModuleList(
    (0): Sequential(
      (0): Conv(
        (conv): Conv2d(192, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): Conv2d(64, 6

In [12]:
# ============================================
# SIMPLE MODEL STRUCTURE CHECK
# ============================================

print("Number of top-level layers:", len(prune_model.model))
print()

for i, layer in enumerate(prune_model.model):
    print(
        f"Layer {i}: "
        f"{layer.__class__.__name__}"
    )

Number of top-level layers: 23

Layer 0: Conv
Layer 1: Conv
Layer 2: C2f
Layer 3: Conv
Layer 4: C2f
Layer 5: Conv
Layer 6: C2f
Layer 7: Conv
Layer 8: C2f
Layer 9: SPPF
Layer 10: Upsample
Layer 11: Concat
Layer 12: C2f
Layer 13: Upsample
Layer 14: Concat
Layer 15: C2f
Layer 16: Conv
Layer 17: Concat
Layer 18: C2f
Layer 19: Conv
Layer 20: Concat
Layer 21: C2f
Layer 22: Detect


In [13]:
# ============================================
# BUILD TORCH-PRUNING DEPENDENCY GRAPH
# ============================================

import torch
import torch_pruning as tp

example_inputs = torch.randn(
    1, 3, 640, 640,
    device=device
)

DG = tp.DependencyGraph()

DG.build_dependency(
    prune_model,
    example_inputs=example_inputs
)

print("Torch-Pruning dependency graph created successfully.")
print("Input shape:", tuple(example_inputs.shape))

Torch-Pruning dependency graph created successfully.
Input shape: (1, 3, 640, 640)


In [14]:
# ============================================
# PREPARE 10% STRUCTURED PRUNING
# ============================================

import torch_pruning as tp

# Identify the detection head
detect_layer = prune_model.model[-1]

print("Protected layer:")
print("Detect:", detect_layer.__class__.__name__)

# Create importance metric
importance = tp.importance.GroupMagnitudeImportance(
    p=2
)

# Layers that must NOT be pruned
ignored_layers = [detect_layer]

print()
print("Pruning target: 10%")
print("Ignored layers:", len(ignored_layers))
print("Importance metric: GroupMagnitudeImportance(p=2)")

Protected layer:
Detect: Detect

Pruning target: 10%
Ignored layers: 1
Importance metric: GroupMagnitudeImportance(p=2)


In [15]:
# ============================================
# CREATE 10% STRUCTURED PRUNER
# ============================================

pruner = tp.pruner.MagnitudePruner(
    model=prune_model,
    example_inputs=example_inputs,
    importance=importance,
    pruning_ratio=0.10,
    ignored_layers=ignored_layers,
)

print("MagnitudePruner created successfully.")
print("Target pruning ratio: 10%")

MagnitudePruner created successfully.
Target pruning ratio: 10%


In [16]:
# ============================================
# APPLY 10% STRUCTURAL PRUNING
# ============================================

before_params = sum(
    p.numel() for p in prune_model.parameters()
)

print("Parameters BEFORE pruning:", before_params)

# Apply the pruning
pruner.step()

after_params = sum(
    p.numel() for p in prune_model.parameters()
)

print("Parameters AFTER pruning :", after_params)

reduction = (
    (before_params - after_params)
    / before_params
) * 100

print()
print(f"Parameter reduction: {reduction:.2f}%")

Parameters BEFORE pruning: 25856899
Parameters AFTER pruning : 25856899

Parameter reduction: 0.00%


In [18]:
# ============================================
# RESET PRUNING COPY
# ============================================

import copy

# Start fresh from the untouched baseline
prune_model = copy.deepcopy(model.model)
prune_model = prune_model.to(device)
prune_model.eval()

print(
    "Fresh pruning copy parameters:",
    sum(p.numel() for p in prune_model.parameters())
)

Fresh pruning copy parameters: 25856899


In [19]:
# ============================================
# INSPECT YOLOv8 C2f IMPLEMENTATION
# ============================================

for i in [2, 4, 6, 8, 12, 15, 18, 21]:
    layer = prune_model.model[i]

    print(f"\nLayer {i}")
    print("Class:", layer.__class__)
    print("cv1:", layer.cv1)
    print("cv2:", layer.cv2)
    print("Number of internal blocks:", len(layer.m))


Layer 2
Class: <class 'ultralytics.nn.modules.block.C2f'>
cv1: Conv(
  (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)
cv2: Conv(
  (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)
Number of internal blocks: 2

Layer 4
Class: <class 'ultralytics.nn.modules.block.C2f'>
cv1: Conv(
  (conv): Conv2d(192, 192, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn): BatchNorm2d(192, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)
cv2: Conv(
  (conv): Conv2d(576, 192, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn): BatchNorm2d(192, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)
Number of internal blocks: 4

Layer 6


In [20]:
# ============================================
# C2f CHANNEL INSPECTION
# ============================================

for i in [2, 4, 6, 8, 12, 15, 18, 21]:
    layer = prune_model.model[i]

    print(
        f"Layer {i}: "
        f"{layer.__class__.__name__} | "
        f"cv1 in={layer.cv1.conv.in_channels}, "
        f"cv1 out={layer.cv1.conv.out_channels} | "
        f"cv2 in={layer.cv2.conv.in_channels}, "
        f"cv2 out={layer.cv2.conv.out_channels} | "
        f"internal blocks={len(layer.m)}"
    )

Layer 2: C2f | cv1 in=96, cv1 out=96 | cv2 in=192, cv2 out=96 | internal blocks=2
Layer 4: C2f | cv1 in=192, cv1 out=192 | cv2 in=576, cv2 out=192 | internal blocks=4
Layer 6: C2f | cv1 in=384, cv1 out=384 | cv2 in=1152, cv2 out=384 | internal blocks=4
Layer 8: C2f | cv1 in=576, cv1 out=576 | cv2 in=1152, cv2 out=576 | internal blocks=2
Layer 12: C2f | cv1 in=960, cv1 out=384 | cv2 in=768, cv2 out=384 | internal blocks=2
Layer 15: C2f | cv1 in=576, cv1 out=192 | cv2 in=384, cv2 out=192 | internal blocks=2
Layer 18: C2f | cv1 in=576, cv1 out=384 | cv2 in=768, cv2 out=384 | internal blocks=2
Layer 21: C2f | cv1 in=960, cv1 out=576 | cv2 in=1152, cv2 out=576 | internal blocks=2


In [21]:
# ============================================
# FIND PRUNABLE CONVOLUTION LAYERS
# ============================================

import torch.nn as nn

for i, layer in enumerate(prune_model.model):
    convs = []

    for name, module in layer.named_modules():
        if isinstance(module, nn.Conv2d):
            convs.append(
                (
                    name,
                    module.in_channels,
                    module.out_channels
                )
            )

    if convs:
        print(f"\nLayer {i} — {layer.__class__.__name__}")

        for name, in_ch, out_ch in convs:
            print(
                f"  {name}: "
                f"in={in_ch}, out={out_ch}"
            )


Layer 0 — Conv
  conv: in=3, out=48

Layer 1 — Conv
  conv: in=48, out=96

Layer 2 — C2f
  cv1.conv: in=96, out=96
  cv2.conv: in=192, out=96
  m.0.cv1.conv: in=48, out=48
  m.0.cv2.conv: in=48, out=48
  m.1.cv1.conv: in=48, out=48
  m.1.cv2.conv: in=48, out=48

Layer 3 — Conv
  conv: in=96, out=192

Layer 4 — C2f
  cv1.conv: in=192, out=192
  cv2.conv: in=576, out=192
  m.0.cv1.conv: in=96, out=96
  m.0.cv2.conv: in=96, out=96
  m.1.cv1.conv: in=96, out=96
  m.1.cv2.conv: in=96, out=96
  m.2.cv1.conv: in=96, out=96
  m.2.cv2.conv: in=96, out=96
  m.3.cv1.conv: in=96, out=96
  m.3.cv2.conv: in=96, out=96

Layer 5 — Conv
  conv: in=192, out=384

Layer 6 — C2f
  cv1.conv: in=384, out=384
  cv2.conv: in=1152, out=384
  m.0.cv1.conv: in=192, out=192
  m.0.cv2.conv: in=192, out=192
  m.1.cv1.conv: in=192, out=192
  m.1.cv2.conv: in=192, out=192
  m.2.cv1.conv: in=192, out=192
  m.2.cv2.conv: in=192, out=192
  m.3.cv1.conv: in=192, out=192
  m.3.cv2.conv: in=192, out=192

Layer 7 — Conv
  c

In [22]:
# ============================================
# TEST ONE STRUCTURAL PRUNING GROUP
# ============================================

import torch_pruning as tp

test_layer = prune_model.model[0].conv

print("Test layer:")
print("Input channels :", test_layer.in_channels)
print("Output channels:", test_layer.out_channels)

# Ask Torch-Pruning for a pruning group
group = DG.get_pruning_group(
    test_layer,
    tp.prune_conv_out_channels,
    idxs=[0]
)

print()
print("Pruning group created:", group)

print()
print("Group is valid:", DG.check_pruning_group(group))

Test layer:
Input channels : 3
Output channels: 48


ValueError: Module Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False) is not in the dependency graph.

In [23]:
# ============================================
# REBUILD DEPENDENCY GRAPH FOR FRESH COPY
# ============================================

import torch_pruning as tp
import torch

example_inputs = torch.randn(
    1, 3, 640, 640,
    device=device
)

DG = tp.DependencyGraph()

DG.build_dependency(
    prune_model,
    example_inputs=example_inputs
)

print("Dependency graph rebuilt successfully.")
print("Graph now belongs to the fresh pruning copy.")

Dependency graph rebuilt successfully.
Graph now belongs to the fresh pruning copy.


In [24]:
# ============================================
# TEST DEPENDENCY GROUP
# ============================================

test_layer = prune_model.model[0].conv

print("Test layer:")
print("Input channels :", test_layer.in_channels)
print("Output channels:", test_layer.out_channels)

group = DG.get_pruning_group(
    test_layer,
    tp.prune_conv_out_channels,
    idxs=[0]
)

print()
print("Pruning group created successfully.")
print("Group is valid:", DG.check_pruning_group(group))

Test layer:
Input channels : 3
Output channels: 48


ValueError: Module Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False) is not in the dependency graph.

In [25]:
# ============================================
# INSPECT TORCH-PRUNING GRAPH
# ============================================

print("Total modules registered in dependency graph:")
print(len(DG.module2node))

print("\nRegistered modules matching Conv2d:")

count = 0

for module in DG.module2node.keys():
    if isinstance(module, torch.nn.Conv2d):
        count += 1

        # Find the module's name inside our model
        module_name = "UNKNOWN"

        for name, candidate in prune_model.named_modules():
            if candidate is module:
                module_name = name
                break

        print(
            f"{count}. {module_name} | "
            f"in={module.in_channels} | "
            f"out={module.out_channels}"
        )

print("\nTotal registered Conv2d modules:", count)

Total modules registered in dependency graph:
1

Registered modules matching Conv2d:
1. model.22.dfl.conv | in=16 | out=1

Total registered Conv2d modules: 1


In [26]:
# ============================================
# REBUILD GRAPH USING ULTRALYTICS FORWARD
# ============================================

import torch
import torch_pruning as tp

# Fresh pruning copy
prune_model = copy.deepcopy(model.model)
prune_model = prune_model.to(device)
prune_model.train()   # important: keep autograd active

# The full Ultralytics DetectionModel is needed
full_model = copy.deepcopy(model.model)

# Actually, use the DetectionModel that owns the layer graph
full_model = copy.deepcopy(model.model)

In [27]:
# ============================================
# CHECK ULTRALYTICS MODEL WRAPPER
# ============================================

print("YOLO wrapper type:")
print(type(model))

print()
print("Underlying model type:")
print(type(model.model))

print()
print("Underlying model forward:")
print(model.model.forward)

print()
print("Number of top-level layers:")
print(len(model.model.model))

YOLO wrapper type:
<class 'ultralytics.models.yolo.model.YOLO'>

Underlying model type:
<class 'ultralytics.nn.tasks.DetectionModel'>

Underlying model forward:
<bound method BaseModel.forward of DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(

In [28]:
print("YOLO wrapper type:")
print(type(model))

print("\nUnderlying model type:")
print(type(model.model))

print("\nInner network type:")
print(type(model.model.model))

YOLO wrapper type:
<class 'ultralytics.models.yolo.model.YOLO'>

Underlying model type:
<class 'ultralytics.nn.tasks.DetectionModel'>

Inner network type:
<class 'torch.nn.modules.container.Sequential'>


In [29]:
# ============================================
# CREATE CORRECT PRUNING COPY
# ============================================

import copy
import torch

# Copy the COMPLETE Ultralytics DetectionModel
prune_model = copy.deepcopy(model.model)

prune_model = prune_model.to(device)
prune_model.eval()

print("Pruning model type:")
print(type(prune_model))

print()
print("Top-level layers:")
print(len(prune_model.model))

print()
print("Parameters:")
print(sum(p.numel() for p in prune_model.parameters()))

Pruning model type:
<class 'ultralytics.nn.tasks.DetectionModel'>

Top-level layers:
23

Parameters:
25856899


In [30]:
# ============================================
# BUILD DEPENDENCY GRAPH — FULL DETECTION MODEL
# ============================================

import torch_pruning as tp
import torch

example_inputs = torch.randn(
    1, 3, 640, 640,
    device=device
)

DG = tp.DependencyGraph()

DG.build_dependency(
    prune_model,
    example_inputs=example_inputs
)

print("Dependency graph built successfully.")
print("Registered modules:", len(DG.module2node))

# Count registered Conv2d layers
conv_count = 0

for module in DG.module2node.keys():
    if isinstance(module, torch.nn.Conv2d):
        conv_count += 1

print("Registered Conv2d layers:", conv_count)

Dependency graph built successfully.
Registered modules: 1
Registered Conv2d layers: 1


In [31]:
# ============================================
# CONVERT C2f -> C2f_v2
# ============================================

import torch
import torch.nn as nn

from ultralytics.nn.modules import C2f, Conv, Bottleneck


# --------------------------------------------
# 1. Define pruning-compatible C2f_v2
# --------------------------------------------

class C2f_v2(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()

        self.c = int(c2 * e)

        self.cv0 = Conv(c1, self.c, 1, 1)
        self.cv1 = Conv(c1, self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)

        self.m = nn.ModuleList(
            Bottleneck(
                self.c,
                self.c,
                shortcut,
                g,
                k=((3, 3), (3, 3)),
                e=1.0
            )
            for _ in range(n)
        )

    def forward(self, x):
        y = [self.cv0(x), self.cv1(x)]
        y.extend(m(y[-1]) for m in self.m)

        return self.cv2(torch.cat(y, 1))


# --------------------------------------------
# 2. Transfer original C2f weights
# --------------------------------------------

def transfer_weights(c2f, c2f_v2):

    c2f_v2.cv2 = c2f.cv2
    c2f_v2.m = c2f.m

    state_dict = c2f.state_dict()
    state_dict_v2 = c2f_v2.state_dict()

    # Original cv1 contains both branches
    old_weight = state_dict["cv1.conv.weight"]

    half_channels = old_weight.shape[0] // 2

    # First half -> cv0
    state_dict_v2["cv0.conv.weight"] = \
        old_weight[:half_channels]

    # Second half -> cv1
    state_dict_v2["cv1.conv.weight"] = \
        old_weight[half_channels:]

    # Transfer BatchNorm parameters
    for bn_key in [
        "weight",
        "bias",
        "running_mean",
        "running_var"
    ]:

        old_bn = state_dict[f"cv1.bn.{bn_key}"]

        state_dict_v2[f"cv0.bn.{bn_key}"] = \
            old_bn[:half_channels]

        state_dict_v2[f"cv1.bn.{bn_key}"] = \
            old_bn[half_channels:]

    # Transfer everything else
    for key in state_dict:

        if not key.startswith("cv1."):
            state_dict_v2[key] = state_dict[key]

    c2f_v2.load_state_dict(state_dict_v2)


# --------------------------------------------
# 3. Replace C2f recursively
# --------------------------------------------

def replace_c2f_with_c2f_v2(module):

    for name, child_module in module.named_children():

        if isinstance(child_module, C2f):

            # Determine shortcut configuration
            shortcut = False

            if len(child_module.m) > 0:
                first_bottleneck = child_module.m[0]
                shortcut = first_bottleneck.add

            c2f_v2 = C2f_v2(
                child_module.cv1.conv.in_channels,
                child_module.cv2.conv.out_channels,
                n=len(child_module.m),
                shortcut=shortcut,
                g=child_module.m[0].cv2.conv.groups,
                e=child_module.c /
                  child_module.cv2.conv.out_channels
            )

            transfer_weights(
                child_module,
                c2f_v2
            )

            setattr(
                module,
                name,
                c2f_v2
            )

        else:
            replace_c2f_with_c2f_v2(child_module)


# --------------------------------------------
# 4. Perform conversion
# --------------------------------------------

replace_c2f_with_c2f_v2(prune_model)

print("C2f conversion completed.")

C2f conversion completed.


In [32]:
# ============================================
# VERIFY C2f -> C2f_v2 CONVERSION
# ============================================

c2f_count = 0
c2f_v2_count = 0

for name, module in prune_model.named_modules():

    if module.__class__.__name__ == "C2f":
        c2f_count += 1

    if module.__class__.__name__ == "C2f_v2":
        c2f_v2_count += 1
        print(
            f"{name} -> C2f_v2 | "
            f"internal blocks: {len(module.m)}"
        )

print()
print("Original C2f remaining:", c2f_count)
print("C2f_v2 modules:", c2f_v2_count)

model.2 -> C2f_v2 | internal blocks: 2
model.4 -> C2f_v2 | internal blocks: 4
model.6 -> C2f_v2 | internal blocks: 4
model.8 -> C2f_v2 | internal blocks: 2
model.12 -> C2f_v2 | internal blocks: 2
model.15 -> C2f_v2 | internal blocks: 2
model.18 -> C2f_v2 | internal blocks: 2
model.21 -> C2f_v2 | internal blocks: 2

Original C2f remaining: 0
C2f_v2 modules: 8


In [33]:
# ============================================
# BUILD PRUNING GRAPH AFTER C2f_v2 CONVERSION
# ============================================

import torch
import torch_pruning as tp

example_inputs = torch.randn(
    1, 3, 640, 640,
    device=device
)

DG = tp.DependencyGraph()

DG.build_dependency(
    prune_model,
    example_inputs=example_inputs
)

print("Dependency graph rebuilt successfully.")
print("Registered modules:", len(DG.module2node))

conv_count = 0

for module in DG.module2node.keys():
    if isinstance(module, torch.nn.Conv2d):
        conv_count += 1

print("Registered Conv2d layers:", conv_count)

AttributeError: 'C2f_v2' object has no attribute 'f'

In [34]:
# ============================================
# RESTORE ULTRALYTICS ROUTING ATTRIBUTES
# ============================================

for name, module in prune_model.named_modules():

    if module.__class__.__name__ == "C2f_v2":

        # Find the corresponding original C2f
        original_module = None

        for original_name, original_candidate in model.model.named_modules():
            if original_name == name:
                original_module = original_candidate
                break

        if original_module is not None:

            # Copy Ultralytics module metadata
            for attr in ["f", "i", "type", "np"]:
                if hasattr(original_module, attr):
                    setattr(
                        module,
                        attr,
                        getattr(original_module, attr)
                    )

print("Ultralytics routing attributes restored.")

Ultralytics routing attributes restored.


In [35]:
# ============================================
# VERIFY ROUTING ATTRIBUTES
# ============================================

missing = []

for name, module in prune_model.named_modules():

    if module.__class__.__name__ == "C2f_v2":

        for attr in ["f", "i", "type"]:
            if not hasattr(module, attr):
                missing.append(f"{name}.{attr}")

if missing:
    print("Missing attributes:")
    for item in missing:
        print(item)
else:
    print("All C2f_v2 routing attributes are present.")

All C2f_v2 routing attributes are present.


In [36]:
# ============================================
# TEST C2f_v2 MODEL FORWARD PASS
# ============================================

import torch

test_input = torch.randn(
    1, 3, 640, 640,
    device=device
)

prune_model.eval()

with torch.no_grad():
    output = prune_model(test_input)

print("Forward pass: SUCCESS")
print("Output type:", type(output))

if isinstance(output, (list, tuple)):
    print("Number of outputs:", len(output))


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [37]:
# ============================================
# MOVE EXPERIMENTAL MODEL TO GPU
# ============================================

prune_model = prune_model.to(device)
prune_model.eval()

print("Model device:", next(prune_model.parameters()).device)

Model device: cuda:0


In [38]:
# ============================================
# TEST C2f_v2 MODEL FORWARD PASS
# ============================================

test_input = torch.randn(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = prune_model(test_input)

print("Forward pass: SUCCESS")
print("Output type:", type(output))

if isinstance(output, (list, tuple)):
    print("Number of outputs:", len(output))

Forward pass: SUCCESS
Output type: <class 'tuple'>
Number of outputs: 2


In [39]:
# ============================================
# STEP 3 — REBUILD TORCH-PRUNING GRAPH
# ============================================

import torch_pruning as tp

prune_model.eval()

example_inputs = torch.randn(
    1, 3, 640, 640,
    device=device
)

DG = tp.DependencyGraph()

print("Building dependency graph...")

DG.build_dependency(
    prune_model,
    example_inputs=example_inputs
)

print("Dependency graph built successfully.")

# Count registered modules
registered_modules = list(DG.module2node.keys())

print("\nRegistered modules:", len(registered_modules))

# Count Conv2d layers visible to Torch-Pruning
registered_convs = [
    m for m in registered_modules
    if isinstance(m, torch.nn.Conv2d)
]

print("Registered Conv2d layers:", len(registered_convs))

print("\nFirst 20 registered Conv2d layers:")
for i, layer in enumerate(registered_convs[:20]):
    print(i, layer)

Building dependency graph...
Dependency graph built successfully.

Registered modules: 307
Registered Conv2d layers: 90

First 20 registered Conv2d layers:
0 Conv2d(192, 1, kernel_size=(1, 1), stride=(1, 1))
1 Conv2d(192, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
2 Conv2d(576, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
3 Conv2d(1152, 576, kernel_size=(1, 1), stride=(1, 1), bias=False)
4 Conv2d(288, 288, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
5 Conv2d(288, 288, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
6 Conv2d(288, 288, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
7 Conv2d(288, 288, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
8 Conv2d(960, 288, kernel_size=(1, 1), stride=(1, 1), bias=False)
9 Conv2d(1152, 576, kernel_size=(1, 1), stride=(1, 1), bias=False)
10 Conv2d(576, 288, kernel_size=(1, 1), stride=(1, 1), bias=False)
11 Conv2d(1152, 576, kernel_size=

In [40]:
# ============================================
# STEP 4 — 10% STRUCTURAL PRUNING EXPERIMENT
# ============================================

import copy
import torch_pruning as tp

# Keep the current converted model as our experimental copy
pruned_model = copy.deepcopy(prune_model)

pruned_model = pruned_model.to(device)
pruned_model.eval()

# Rebuild dependency graph for the copy
example_inputs = torch.randn(
    1, 3, 640, 640,
    device=device
)

DG_pruned = tp.DependencyGraph()

DG_pruned.build_dependency(
    pruned_model,
    example_inputs=example_inputs
)

print("Dependency graph ready.")

# ------------------------------------------------
# Identify the detection head
# ------------------------------------------------

detect_layer = pruned_model.model[-1]

print("Detection head:", type(detect_layer))

# ------------------------------------------------
# Create pruning importance criterion
# ------------------------------------------------

importance = tp.importance.GroupMagnitudeImportance(
    p=2
)

# ------------------------------------------------
# Create the pruner
# ------------------------------------------------

pruner = tp.pruner.MetaPruner(
    pruned_model,
    example_inputs=example_inputs,
    importance=importance,
    pruning_ratio=0.10,
    ignored_layers=[detect_layer],
)

print("Pruner created.")
print("Starting 10% structural pruning...")

# ------------------------------------------------
# Apply pruning
# ------------------------------------------------

pruner.step()

print("Pruning completed.")

Dependency graph ready.
Detection head: <class 'ultralytics.nn.modules.head.Detect'>
Pruner created.
Starting 10% structural pruning...
Pruning completed.


In [41]:
# ============================================
# STEP 4B — CHECK WHETHER PRUNING ACTUALLY
# REDUCED THE MODEL
# ============================================

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

original_params = count_parameters(prune_model)
pruned_params = count_parameters(pruned_model)

reduction = (
    (original_params - pruned_params)
    / original_params
    * 100
)

print("Original experimental model parameters:")
print(f"{original_params:,}")

print("\nPruned model parameters:")
print(f"{pruned_params:,}")

print(f"\nParameter reduction: {reduction:.2f}%")

Original experimental model parameters:
25,856,899

Pruned model parameters:
21,317,894

Parameter reduction: 17.55%


In [42]:
# ============================================
# STEP 5 — TEST PRUNED MODEL FORWARD PASS
# ============================================

pruned_model = pruned_model.to(device)
pruned_model.eval()

test_input = torch.randn(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = pruned_model(test_input)

print("Pruned model forward pass: SUCCESS")
print("Output type:", type(output))

if isinstance(output, (list, tuple)):
    print("Number of outputs:", len(output))

Pruned model forward pass: SUCCESS
Output type: <class 'tuple'>
Number of outputs: 2


In [43]:
# ============================================
# STEP 6 — MEASURE PRUNED MODEL
# ============================================

import time
import torch

pruned_model = pruned_model.to(device)
pruned_model.eval()

# --------------------------------------------
# Parameter count
# --------------------------------------------

pruned_params = sum(
    p.numel() for p in pruned_model.parameters()
)

print(f"Parameters: {pruned_params:,}")

# --------------------------------------------
# FLOPs / MACs
# --------------------------------------------

example_input = torch.randn(
    1, 3, 640, 640,
    device=device
)

try:
    macs, params = tp.utils.count_ops_and_params(
        pruned_model,
        example_input
    )

    flops = macs * 2

    print(f"MACs: {macs / 1e9:.3f} G")
    print(f"FLOPs: {flops / 1e9:.3f} G")

except Exception as e:
    print("FLOPs/MACs measurement failed:")
    print(e)

# --------------------------------------------
# GPU warm-up
# --------------------------------------------

with torch.no_grad():
    for _ in range(20):
        _ = pruned_model(example_input)

torch.cuda.synchronize()

# --------------------------------------------
# Timed inference
# --------------------------------------------

num_runs = 100

start = time.perf_counter()

with torch.no_grad():
    for _ in range(num_runs):
        _ = pruned_model(example_input)

torch.cuda.synchronize()

elapsed = time.perf_counter() - start

avg_ms = (elapsed / num_runs) * 1000
fps = 1000 / avg_ms

print(f"\nAverage inference: {avg_ms:.2f} ms/image")
print(f"FPS: {fps:.2f}")

Parameters: 21,317,894
MACs: 33.030 G
FLOPs: 66.060 G

Average inference: 21.58 ms/image
FPS: 46.33
